# Phase 1 EDA — Tweet Style Clustering

Loads the cleaned tweet corpus, embeds it, clusters by style, and visualizes the clusters.
This is the screenshot-worthy deliverable for Phase 1.

In [ ]:
import sys
sys.path.append('../backend')

import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from preprocessing import load_raw_tweets, clean_tweets
from clustering import embed_tweets, cluster_embeddings, EMBEDDING_MODEL
from sentence_transformers import SentenceTransformer

## Load + clean raw tweets

In [ ]:
raw = load_raw_tweets('../backend/data/raw_tweets.json')
cleaned = clean_tweets(raw)
print(f"raw: {len(raw)}  cleaned: {len(cleaned)}")
pd.DataFrame(cleaned).head()

## Embed + cluster

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embed_tweets(cleaned, model)
labels = cluster_embeddings(embeddings, n_clusters=4)

df = pd.DataFrame(cleaned)
df['cluster'] = labels
df['cluster'].value_counts().sort_index()

## Sample tweets per cluster (sanity check the style buckets)

In [ ]:
for cluster_id in sorted(df['cluster'].unique()):
    print(f"\n--- cluster {cluster_id} ---")
    for text in df[df['cluster'] == cluster_id]['text'].head(5):
        print(' -', text)

## 2D visualization of the clusters

In [ ]:
coords = PCA(n_components=2, random_state=42).fit_transform(embeddings)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10')
plt.title('Tweet style clusters (PCA projection)')
plt.colorbar(scatter, label='cluster')
plt.show()